In [2]:
import os
print(os.getcwd())

/Users/Mourya/projects/aircraft-conflict-gnn/notebooks


In [4]:
from pathlib import Path

print("Current directory:")
print(Path.cwd())

print("\nContents of current directory:")
for p in Path(".").iterdir():
    print(p)

Current directory:
/Users/Mourya/projects/aircraft-conflict-gnn/notebooks

Contents of current directory:
01_conflict_prediction_v1.ipynb
01_load_adsb_trace.ipynb
graphs.pt
models
processed_data
04_train_gcn.ipynb
.ipynb_checkpoints
conflict_gat_v1.pth


In [5]:
from pathlib import Path

print("processed_data exists:", Path("processed_data").exists())

print("\nContents of processed_data:")

for f in sorted(Path("processed_data").iterdir()):
    print(f.name)

processed_data exists: True

Contents of processed_data:
graphs_large.pt
graphs_v1.pt
labelled_large.csv
labelled_pairs_v1.csv
pairs_df_v1.csv
region_final_v1.csv
region_large.csv
test_graphs.pt
train_graphs.pt
val_graphs.pt


In [1]:
import json
from pathlib import Path
import pandas as pd

print("Notebook is working!")

Notebook is working!


In [2]:
from pathlib import Path

file_path = Path("../data/raw/adsbx/trace_full_~2c2101.json")

print("Exists:", file_path.exists())
print("Absolute path:", file_path.resolve())

Exists: False
Absolute path: /Users/Mourya/projects/data/raw/adsbx/trace_full_~2c2101.json


In [3]:
with open(file_path, "r") as f:
    data = json.load(f)

print(type(data))
print(data.keys())

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/adsbx/trace_full_~2c2101.json'

In [ ]:
for key, value in data.items():
    if isinstance(value, list):
        print(f"{key}: list ({len(value)} items)")
    else:
        print(f"{key}: {value}")

In [ ]:
trace = data["trace"]

print("Number of trajectory points:", len(trace))
print("\nFirst trajectory point:")
print(trace[0])

In [ ]:
trace = data["trace"]

print("Number of points:", len(trace))
print("\nFirst point:")
print(trace[0])

In [ ]:
import pandas as pd

df = pd.DataFrame(trace)

print(df.shape)
df.head()

In [ ]:
columns = [
    "time_offset",
    "latitude",
    "longitude",
    "baro_altitude",
    "ground_speed",
    "track",
    "quality",
    "vertical_rate",
    "metadata",
    "source",
    "geo_altitude",
    "geo_vertical_rate",
    "reserved1",
    "reserved2"
]

df = pd.DataFrame(trace, columns=columns)

df.head()

In [ ]:
df["timestamp"] = pd.to_datetime(
    data["timestamp"] + df["time_offset"],
    unit="s",
    utc=True
)

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Keep only the columns needed for modeling
flight_df = df[
    [
        "timestamp",
        "latitude",
        "longitude",
        "baro_altitude",
        "ground_speed",
        "track",
        "vertical_rate",
    ]
].copy()

# Remove rows with missing values
flight_df = flight_df.dropna().reset_index(drop=True)

flight_df.head()

In [ ]:
flight_df.info()

In [ ]:
import fastavro
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/opensky_avro/states_2017-06-05-02.avro")
print("Exists:", file_path.exists())

with open(file_path, "rb") as f:
    records = list(fastavro.reader(f))

df = pd.DataFrame(records)
print(df.shape)
print(df.columns)
df.head()

In [ ]:
from pathlib import Path

file_path = Path("../data/raw/opensky_avro/states_2017-06-05-02.avro")

print(file_path.exists())

In [ ]:
import fastavro
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/opensky_avro/states_2017-06-05-02.avro")

print("Exists:", file_path.exists())

with open(file_path, "rb") as f:
    records = list(fastavro.reader(f))

df = pd.DataFrame(records)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

In [ ]:
import fastavro
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/opensky_avro/states_2017-06-05-02.avro")

with open(file_path, "rb") as f:
    records = list(fastavro.reader(f))

df = pd.DataFrame(records)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

In [ ]:
import fastavro
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/opensky_avro/states_2017-06-05-02.avro")
print("Exists:", file_path.exists())

with open(file_path, "rb") as f:
    records = list(fastavro.reader(f))

df = pd.DataFrame(records)
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
print(df['lat'].describe())
print(df['lon'].describe())

In [ ]:
region = df[(df['lat'].between(48, 53)) & (df['lon'].between(5, 15))]
print(region.shape)
print(region['icao24'].nunique(), "unique aircraft")

In [ ]:
counts = region.groupby('time')['icao24'].nunique()
print(counts.describe())

In [ ]:
region['datetime'] = pd.to_datetime(region['time'], unit='s', utc=True)

In [ ]:
# Rebuild region_final from region

region_final = region[
    (region["onground"] == False) &
    (region["baroaltitude"] > 1000)
].copy()

region_final = region_final.sort_values(["icao24", "time"])

region_final["dt"] = region_final.groupby("icao24")["time"].diff()
region_final["dalt"] = region_final.groupby("icao24")["baroaltitude"].diff()

region_final["climb_rate_fpm"] = (
    region_final["dalt"] / region_final["dt"] * 60
)

MAX_CLIMB_RATE = 6000

region_final = region_final[
    (region_final["climb_rate_fpm"].abs() <= MAX_CLIMB_RATE) |
    (region_final["climb_rate_fpm"].isna())
].copy()

print("region_final:", region_final.shape)
print("Aircraft:", region_final["icao24"].nunique())

In [ ]:
from itertools import combinations
import numpy as np

pair_records = []

for t, group in region_final.groupby("time"):

    aircraft = group.set_index("icao24")
    ids = aircraft.index.tolist()

    for id1, id2 in combinations(ids, 2):

        a1 = aircraft.loc[id1]
        a2 = aircraft.loc[id2]

        h_dist = haversine_nm(
            a1["lat"], a1["lon"],
            a2["lat"], a2["lon"]
        )

        v_dist = abs(
            a1["baroaltitude"] -
            a2["baroaltitude"]
        ) * 3.28084

        pair_records.append({
            "time": t,
            "icao24_1": id1,
            "icao24_2": id2,
            "h_dist_nm": h_dist,
            "v_dist_ft": v_dist
        })

pairs_df = pd.DataFrame(pair_records)

print("pairs_df:", pairs_df.shape)

In [ ]:
pairs_df[["icao24_1", "icao24_2"]] = np.sort(
    pairs_df[["icao24_1", "icao24_2"]],
    axis=1
)

pairs_df = (
    pairs_df
    .sort_values(["icao24_1", "icao24_2", "time"])
    .reset_index(drop=True)
)

pairs_df["h_dist_prev"] = (
    pairs_df
    .groupby(["icao24_1", "icao24_2"])["h_dist_nm"]
    .shift(1)
)

pairs_df["time_prev"] = (
    pairs_df
    .groupby(["icao24_1", "icao24_2"])["time"]
    .shift(1)
)

pairs_df["closing_rate_nm_s"] = (
    pairs_df["h_dist_prev"] - pairs_df["h_dist_nm"]
) / (
    pairs_df["time"] - pairs_df["time_prev"]
)

dt = pairs_df["time"] - pairs_df["time_prev"]

pairs_df.loc[dt <= 0, "closing_rate_nm_s"] = np.nan
pairs_df.loc[
    pairs_df["closing_rate_nm_s"].abs() > 5,
    "closing_rate_nm_s"
] = np.nan

print(pairs_df.head())

In [ ]:
labelled = will_conflict_within(
    pairs_df,
    horizon_s=300
)

print(
    "Positive labels:",
    labelled["label_conflict"].sum()
)

print(labelled.head())

In [ ]:
import torch
from torch_geometric.data import Data

node_feature_cols = [
    "lat",
    "lon",
    "baroaltitude",
    "velocity",
    "heading",
    "vertrate",
]

graphs = []

for t in sorted(region_final["time"].unique()):

    nodes = region_final[region_final["time"] == t].copy()
    edges = labelled[labelled["time"] == t].copy()

    if len(nodes) < 2 or len(edges) == 0:
        continue

    node_map = {
        icao: idx
        for idx, icao in enumerate(nodes["icao24"])
    }

    edges = edges[
        edges["icao24_1"].isin(node_map)
        & edges["icao24_2"].isin(node_map)
    ].copy()

    if len(edges) == 0:
        continue

    edge_index = torch.tensor(
        [
            [node_map[a] for a in edges["icao24_1"]],
            [node_map[b] for b in edges["icao24_2"]],
        ],
        dtype=torch.long,
    )

    x = torch.tensor(
        nodes[node_feature_cols]
        .fillna(0)
        .values,
        dtype=torch.float,
    )

    edge_attr = torch.tensor(
        edges[
            [
                "h_dist_nm",
                "v_dist_ft",
                "closing_rate_nm_s",
            ]
        ]
        .fillna(0)
        .values,
        dtype=torch.float,
    )

    y = torch.tensor(
        edges["label_conflict"]
        .astype(int)
        .values,
        dtype=torch.long,
    )

    graph = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=y,
    )

    graph.time = int(t)

    graphs.append(graph)

In [ ]:
print(f"Graph snapshots: {len(graphs)}")

if len(graphs) > 0:
    g = graphs[0]

    print(g)
    print(f"Nodes: {g.num_nodes}")
    print(f"Edges: {g.edge_index.shape[1]}")
    print(f"Node feature dimension: {g.x.shape[1]}")
    print(f"Edge feature dimension: {g.edge_attr.shape[1]}")
    print(f"Positive edges: {int(g.y.sum())}")

In [ ]:
total_nodes = sum(g.num_nodes for g in graphs)
total_edges = sum(g.edge_index.shape[1] for g in graphs)
total_positive = sum(int(g.y.sum()) for g in graphs)

print("Dataset Summary")
print("----------------")
print(f"Graphs: {len(graphs)}")
print(f"Total nodes: {total_nodes}")
print(f"Total edges: {total_edges}")
print(f"Positive conflict edges: {total_positive}")
print(f"Positive rate: {100 * total_positive / total_edges:.4f}%")

In [ ]:
from torch_geometric.loader import DataLoader

n = len(graphs)

train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_graphs = graphs[:train_end]
val_graphs = graphs[train_end:val_end]
test_graphs = graphs[val_end:]

print(f"Train graphs: {len(train_graphs)}")
print(f"Validation graphs: {len(val_graphs)}")
print(f"Test graphs: {len(test_graphs)}")

train_loader = DataLoader(train_graphs, batch_size=8, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=8, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=8, shuffle=False)

In [ ]:
import torch

num_positive = sum(int(g.y.sum()) for g in train_graphs)
num_total = sum(g.y.numel() for g in train_graphs)
num_negative = num_total - num_positive

print("Training positives:", num_positive)
print("Training negatives:", num_negative)

pos_weight = torch.tensor(
    [num_negative / max(num_positive, 1)],
    dtype=torch.float,
)

print("pos_weight =", pos_weight.item())

In [ ]:
import torch
import torch.nn.functional as F

from torch import nn
from torch_geometric.nn import GATConv


class ConflictGAT(nn.Module):

    def __init__(
        self,
        node_dim=6,
        edge_dim=3,
        hidden_dim=64,
    ):
        super().__init__()

        self.gat1 = GATConv(
            node_dim,
            hidden_dim,
            heads=2,
            edge_dim=edge_dim,
            concat=True,
        )

        self.gat2 = GATConv(
            hidden_dim * 2,
            hidden_dim,
            heads=1,
            edge_dim=edge_dim,
            concat=False,
        )

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, data):

        x = self.gat1(
            data.x,
            data.edge_index,
            data.edge_attr,
        )

        x = F.relu(x)

        x = self.gat2(
            x,
            data.edge_index,
            data.edge_attr,
        )

        src = x[data.edge_index[0]]
        dst = x[data.edge_index[1]]

        edge_input = torch.cat(
            [
                src,
                dst,
                data.edge_attr,
            ],
            dim=1,
        )

        logits = self.edge_mlp(edge_input)

        return logits.squeeze(-1)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

model = ConflictGAT().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight.to(device)
)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def train_epoch(model, loader, optimizer, criterion, device):

    model.train()

    total_loss = 0.0

    for batch in loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        logits = model(batch)

        loss = criterion(
            logits,
            batch.y.float()
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate(model, loader, criterion, device):

    model.eval()

    losses = []

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():

        for batch in loader:

            batch = batch.to(device)

            logits = model(batch)

            loss = criterion(
                logits,
                batch.y.float()
            )

            losses.append(loss.item())

            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).long()

            y_true.extend(batch.y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    try:
        roc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc = float("nan")

    return {
        "loss": np.mean(losses),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc,
    }

In [ ]:
NUM_EPOCHS = 20

best_f1 = 0.0
best_state = None

history = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device,
    )

    val_metrics = evaluate(
        model,
        val_loader,
        criterion,
        device,
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train {train_loss:.4f} | "
        f"Val {val_metrics['loss']:.4f} | "
        f"F1 {val_metrics['f1']:.4f} | "
        f"Recall {val_metrics['recall']:.4f} | "
        f"Precision {val_metrics['precision']:.4f}"
    )

    if val_metrics["f1"] > best_f1:

        best_f1 = val_metrics["f1"]

        best_state = {
            k: v.cpu()
            for k, v in model.state_dict().items()
        }

print("\nBest validation F1:", best_f1)

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

test_metrics = evaluate(
    model,
    test_loader,
    criterion,
    device,
)

print("\nTest Metrics")
for k, v in test_metrics.items():
    print(f"{k:12s}: {v:.4f}")

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

model.eval()

val_probs = []
val_labels = []

with torch.no_grad():

    for batch in val_loader:

        batch = batch.to(device)

        logits = model(batch)

        probs = torch.sigmoid(logits)

        val_probs.extend(probs.cpu().numpy())
        val_labels.extend(batch.y.cpu().numpy())

val_probs = np.array(val_probs)
val_labels = np.array(val_labels)

print("Validation samples:", len(val_labels))
print("Positive labels:", val_labels.sum())

In [ ]:
from sklearn.metrics import f1_score

thresholds = np.arange(0.01, 1.00, 0.01)

best_threshold = 0.5
best_f1 = -1

results = []

for t in thresholds:

    preds = (val_probs >= t).astype(int)

    f1 = f1_score(
        val_labels,
        preds,
        zero_division=0
    )

    results.append((t, f1))

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best validation F1:", best_f1)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

model.eval()

test_probs = []
test_labels = []

with torch.no_grad():

    for batch in test_loader:

        batch = batch.to(device)

        logits = model(batch)

        probs = torch.sigmoid(logits)

        test_probs.extend(probs.cpu().numpy())
        test_labels.extend(batch.y.cpu().numpy())

test_probs = np.array(test_probs)
test_labels = np.array(test_labels)

test_preds = (test_probs >= best_threshold).astype(int)

print("Threshold:", best_threshold)
print("Precision:", precision_score(test_labels, test_preds, zero_division=0))
print("Recall:", recall_score(test_labels, test_preds, zero_division=0))
print("F1:", f1_score(test_labels, test_preds, zero_division=0))
print("ROC-AUC:", roc_auc_score(test_labels, test_probs))

print("\nConfusion Matrix")
print(confusion_matrix(test_labels, test_preds))

In [ ]:
import matplotlib.pyplot as plt

thresholds = [x[0] for x in results]
f1_scores = [x[1] for x in results]

plt.figure(figsize=(8,5))
plt.plot(thresholds, f1_scores)
plt.xlabel("Decision Threshold")
plt.ylabel("Validation F1")
plt.title("Threshold Optimization")
plt.grid(True)
plt.show()

In [ ]:
torch.save(model.state_dict(), "conflict_gat_v1.pth")
torch.save(graphs, "graphs.pt")

In [ ]:
import os
import torch

os.makedirs("models", exist_ok=True)

torch.save(model.state_dict(), "models/conflict_gat_v1.pth")

print("Model saved successfully.")

In [ ]:
print(model)

In [4]:
import os
print(os.getcwd())

/Users/Mourya/projects/aircraft-conflict-gnn


In [5]:
import os

file_path = "../data/raw/adsbx/trace_full_~2c2101.json"

print(os.path.exists(file_path))

False


In [6]:
from pathlib import Path

for p in Path(".").rglob("trace_full_~2c2101.json"):
    print(p)

trace_full_~2c2101.json
data/raw/adsbx/trace_full_~2c2101.json


In [7]:
import os
print(os.getcwd())

/Users/Mourya/projects/aircraft-conflict-gnn


In [8]:
from pathlib import Path

file_path = Path("data/raw/adsbx/trace_full_~2c2101.json")

with open(file_path, "r") as f:
    data = json.load(f)

In [9]:
print("model" in globals())
print("graphs" in globals())
print("labelled" in globals())
print("pairs_df" in globals())
print("region_final" in globals())

False
False
False
False
False
